# 🔬 Day 5: First Inference & Automatic Evaluation
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Jira Task:** `KAN-30`
### **Models:** `qwen_sme_v1` | `llama_sme_v1`
### **Hardware:** Google Colab T4 GPU (15 GB VRAM)

## Cell 1 — Mount Drive & GPU Check

In [ ]:
import os, json, gc, torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

QWEN_ADAPTER_DIR  = os.path.join(PROJECT_ROOT, 'models', 'v1', 'qwen_sme_v1')
LLAMA_ADAPTER_DIR = os.path.join(PROJECT_ROOT, 'models', 'v1', 'llama_sme_v1')
RESULTS_DIR       = os.path.join(PROJECT_ROOT, 'evaluation', 'day5')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Verify adapter dirs exist
for name, path in [('Qwen adapter', QWEN_ADAPTER_DIR), ('Llama adapter', LLAMA_ADAPTER_DIR)]:
    exists = os.path.exists(os.path.join(path, 'adapter_config.json'))
    print(f"{'✅' if exists else '❌'} {name}: {path}")
    if not exists:
        raise FileNotFoundError(f"{name} not found at {path}. Run Day 4 first.")

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"✅ GPU : {torch.cuda.get_device_name(0)} | {free/(1024**3):.1f} GB free / {total/(1024**3):.1f} GB total")
else:
    raise RuntimeError("❌ No GPU — Runtime → Change runtime type → T4 GPU")

## Cell 2 — Install Libraries

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft
!pip install -q rouge-score sacrebleu bert-score
import transformers, peft
print(f"Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("✅ Libraries ready.")

## Cell 3 — Hugging Face Login

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("✅ HF logged in via Colab Secret.")
except Exception:
    print("ℹ️ Continuing with public HF access.")

## Cell 4 — Load Test Dataset

In [ ]:
# FIX: Try test_v1.json first, fall back to val_v1.json if test doesn't exist
test_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_v1.json')
val_path  = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v1.json')

if os.path.exists(test_path):
    with open(test_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    print(f"✅ Loaded test_v1.json: {len(test_data):,} samples")
elif os.path.exists(val_path):
    with open(val_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    print(f"ℹ️  test_v1.json not found — using val_v1.json: {len(test_data):,} samples")
else:
    raise FileNotFoundError("Neither test_v1.json nor val_v1.json found. Run Day 2 first.")

# Use up to 50 samples (fast on T4, representative)
eval_samples = test_data[:50]
references   = [ex['response'] for ex in eval_samples]
print(f"Using {len(eval_samples)} samples for evaluation.")
print(f"Sample keys: {list(eval_samples[0].keys())}")

## Cell 5 — Helper Functions

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bert_score_fn


def load_model_with_adapter(base_model_id, adapter_dir):
    """Load 4-bit base model and attach fine-tuned LoRA adapter."""
    print(f"  Loading tokenizer from adapter dir...")
    # FIX: load tokenizer from adapter_dir (has correct special tokens)
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'   # left-pad for generation

    print(f"  Loading base model: {base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        ),
        device_map='auto',
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    print(f"  Attaching LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  ✅ Ready. VRAM: {free/(1024**3):.1f} GB free")
    return model, tokenizer


def build_prompt(ex, tokenizer):
    """Build ChatML prompt without assistant response (for inference)."""
    sys_msg = 'You are an expert SME daily business assistant.'
    user_q  = ex['instruction']
    if ex.get('context'):
        user_q = f"Context:\n{ex['context']}\n\nQuestion:\n{ex['instruction']}"
    # FIX: add_generation_prompt=True appends the assistant turn start token
    return tokenizer.apply_chat_template(
        [{'role':'system','content':sys_msg},
         {'role':'user',  'content':user_q}],
        tokenize=False,
        add_generation_prompt=True
    )


@torch.no_grad()
def run_inference(model, tokenizer, samples, max_new_tokens=200, batch_size=2):
    """Run batched inference. Returns list of generated strings."""
    predictions = []
    for i in range(0, len(samples), batch_size):
        batch   = samples[i:i+batch_size]
        prompts = [build_prompt(ex, tokenizer) for ex in batch]
        inputs  = tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=512
        ).to('cuda')
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        for j, out in enumerate(outputs):
            input_len = inputs['input_ids'].shape[1]
            decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
            predictions.append(decoded)
        print(f"  {min(i+batch_size, len(samples))}/{len(samples)} done...", end='\r')
    print()
    return predictions


def compute_metrics(predictions, references):
    """ROUGE-1/2/L, BLEU-4, BERTScore."""
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1 = r2 = rl = 0.0
    for pred, ref in zip(predictions, references):
        s   = scorer.score(ref, pred)
        r1 += s['rouge1'].fmeasure
        r2 += s['rouge2'].fmeasure
        rl += s['rougeL'].fmeasure
    n = len(predictions)

    bleu = sacrebleu.corpus_bleu(predictions, [references]).score

    # FIX: force BERTScore to CPU to avoid VRAM OOM during eval
    print("  Computing BERTScore on CPU (~1 min)...")
    _, _, F = bert_score_fn(
        predictions, references,
        lang='en',
        model_type='distilbert-base-uncased',
        device='cpu',      # ← forced CPU: avoids OOM when model is on GPU
        verbose=False
    )
    return {
        'rouge1':    round(r1/n * 100, 2),
        'rouge2':    round(r2/n * 100, 2),
        'rougeL':    round(rl/n * 100, 2),
        'bleu4':     round(bleu, 2),
        'bertscore': round(F.mean().item() * 100, 2)
    }


print('✅ Helper functions loaded.')

## Cell 6 — Evaluate Qwen 2.5-7B (`qwen_sme_v1`)

In [ ]:
gc.collect(); torch.cuda.empty_cache()

print("="*55 + "\n  EVALUATING: qwen_sme_v1\n" + "="*55)
qwen_model, qwen_tok = load_model_with_adapter(
    'Qwen/Qwen2.5-7B-Instruct', QWEN_ADAPTER_DIR
)

print("\nRunning inference...")
qwen_preds = run_inference(qwen_model, qwen_tok, eval_samples)

print("Computing metrics...")
qwen_scores = compute_metrics(qwen_preds, references)

print("\n📊 Qwen 2.5-7B Results:")
for k, v in qwen_scores.items():
    print(f"   {k:<12}: {v}")

print("\n--- Sample Predictions (first 2) ---")
for i in range(min(2, len(eval_samples))):
    print(f"\n[{i+1}] Q  : {eval_samples[i]['instruction'][:100]}")
    print(f"     Ref: {references[i][:150]}")
    print(f"     Gen: {qwen_preds[i][:150]}")

del qwen_model
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
free, _ = torch.cuda.mem_get_info()
print(f"\n🧹 Qwen unloaded. {free/(1024**3):.1f} GB VRAM free for Llama.")

## Cell 7 — Evaluate Llama 3 8B (`llama_sme_v1`)

In [ ]:
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()

print("="*55 + "\n  EVALUATING: llama_sme_v1\n" + "="*55)
llama_model, llama_tok = load_model_with_adapter(
    'meta-llama/Meta-Llama-3-8B-Instruct', LLAMA_ADAPTER_DIR
)

print("\nRunning inference...")
llama_preds = run_inference(llama_model, llama_tok, eval_samples)

print("Computing metrics...")
llama_scores = compute_metrics(llama_preds, references)

print("\n📊 Llama 3 8B Results:")
for k, v in llama_scores.items():
    print(f"   {k:<12}: {v}")

print("\n--- Sample Predictions (first 2) ---")
for i in range(min(2, len(eval_samples))):
    print(f"\n[{i+1}] Q  : {eval_samples[i]['instruction'][:100]}")
    print(f"     Ref: {references[i][:150]}")
    print(f"     Gen: {llama_preds[i][:150]}")

del llama_model
gc.collect(); torch.cuda.empty_cache()
print("\n🧹 Llama unloaded.")

## Cell 8 — Comparison Table & Save Results

In [ ]:
print("\n" + "="*65)
print("  📊 DAY 5 — Qwen 2.5-7B vs Llama 3 8B")
print("="*65)
print(f"{'Metric':<14} {'Qwen 2.5-7B':>14} {'Llama 3 8B':>12} {'Winner':>10}")
print("-"*65)
for metric in ['rouge1','rouge2','rougeL','bleu4','bertscore']:
    q = qwen_scores[metric]
    l = llama_scores[metric]
    winner = '🟢 Qwen' if q >= l else '🟡 Llama'
    print(f"{metric:<14} {q:>14.2f} {l:>12.2f} {winner:>10}")
print("="*65)

results = {
    'Day': 'Day 5 - First Inference & Evaluation',
    'Jira_Task': 'KAN-30',
    'Domain': 'SME Daily Business',
    'eval_samples': len(eval_samples),
    'Scores': {
        'qwen_sme_v1':  qwen_scores,
        'llama_sme_v1': llama_scores
    },
    'Sample_Predictions': [
        {
            'question':   eval_samples[i]['instruction'],
            'reference':  references[i],
            'qwen_pred':  qwen_preds[i],
            'llama_pred': llama_preds[i]
        }
        for i in range(min(10, len(eval_samples)))
    ]
}

results_path = os.path.join(RESULTS_DIR, 'day5_evaluation_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

meta_path = os.path.join(PROJECT_ROOT, 'day5_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        'Day': 'Day 5 - First Inference & Evaluation',
        'Jira_Task': 'KAN-30',
        'Domain': 'SME Daily Business',
        'eval_samples': len(eval_samples),
        'qwen_sme_v1':  qwen_scores,
        'llama_sme_v1': llama_scores
    }, f, indent=2)

print(f"\n✅ Results saved  : {results_path}")
print(f"✅ Metadata saved : {meta_path}")
print("\n🎉 Ready for Day 6: Iterative Fine-Tuning & Optimization (KAN-34)!")